## Photogenic Value Index

# Bagian Baru

In [ ]:
!pip install transformers torch torchvision Pillow matplotlib numpy scikit-image


### Catatan perbaikan teknis
Seluruh perhitungan kelas sekarang menggunakan `class_masks` (mask biner per kelas), bukan membandingkan kanal merah dari `mask_final`. `mask_final` hanya digunakan untuk visualisasi. Mask kelas yang tumpang tindih setelah operasi morfologi diselesaikan dengan prioritas deterministik agar setiap piksel hanya dihitung pada satu kelas. Tahapan inferensi SegFormer, morfologi, fitur morfologi/RGB/GLCM, delapan indeks, bobot teoretis, grafik, dan ekspor PDF tetap dipertahankan.


## Analisis Multi-Foto
Pada sel pemrosesan utama, gunakan tombol upload untuk memilih beberapa foto sekaligus (JPG/JPEG/PNG/BMP/WEBP). Model SegFormer dimuat satu kali, lalu setiap foto diproses melalui alur lengkap yang sama. Setiap foto menghasilkan laporan PDF tersendiri; seluruh PDF dikemas ke dalam satu ZIP untuk diunduh. Nama file laporan mengikuti nama masing-masing foto.

In [ ]:
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import numpy as np
import cv2

# ===============================
# 1. LOAD MODEL
# ===============================
model_name = "nvidia/segformer-b5-finetuned-cityscapes-1024-1024"
processor = SegformerImageProcessor.from_pretrained(model_name)
model = SegformerForSemanticSegmentation.from_pretrained(model_name)
model.eval()


# ============================================================
# INPUT MULTI-FOTO: unggah beberapa gambar sekaligus
# ============================================================
from google.colab import files
import os
import zipfile

uploaded = files.upload()  # Pilih lebih dari satu foto sekaligus
uploaded_image_paths = [
    name for name in uploaded.keys()
    if name.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
]
if not uploaded_image_paths:
    raise ValueError("Tidak ada file citra yang didukung. Unggah JPG, JPEG, PNG, BMP, atau WEBP.")

print(f"Jumlah foto yang akan dianalisis: {len(uploaded_image_paths)}")
generated_pdfs = []

for image_index, uploaded_image_path in enumerate(uploaded_image_paths, start=1):
    print("\\n" + "="*90)
    print(f"ANALISIS FOTO {image_index}/{len(uploaded_image_paths)}: {uploaded_image_path}")
    print("="*90)

    # ===============================
    # 2. LOAD IMAGE
    # ===============================
    image_path = uploaded_image_path
    image = Image.open(image_path).convert("RGB")
    img = np.array(image)

    # ===============================
    # 4. SEGMENTASI AI
    # ===============================
    inputs = processor(images=image, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    upsampled_logits = torch.nn.functional.interpolate(
        logits,
        size=image.size[::-1],
        mode="bilinear",
        align_corners=False
    )

    pred_seg = upsampled_logits.argmax(dim=1)[0].cpu().numpy()

    h, w = pred_seg.shape

    # ===============================
    # 5. MAPPING CLASS
    # ===============================
    def remap_class(class_id):
        if class_id in [0, 1]: return "ground"
        elif class_id == 2: return "building"
        elif class_id in [8, 9]: return "vegetation"
        elif class_id == 10: return "sky"
        elif class_id in [13, 14, 15]: return "vehicle_4w"
        elif class_id in [17, 18]: return "vehicle_2w"
        elif class_id in [11, 12]: return "human"
        elif class_id in [6, 7]: return "traffic_sign"
        else: return None

    COLORS = {
        "ground": [150, 100, 100],
        "building": [70, 70, 70],
        "vegetation": [107, 142, 35],
        "sky": [70, 130, 180],
        "vehicle_4w": [0, 0, 142],
        "vehicle_2w": [0, 0, 230],
        "human": [220, 20, 60],
        "traffic_sign": [220, 220, 0]
    }

    # ===============================
    # 6. SEGMENTASI CLEAN (GLOBAL)
    # ===============================
    # PENTING: class_masks menyimpan mask biner per kelas, terpisah dari
    # mask_final yang hanya dipakai untuk visualisasi. Ini mencegah tabrakan
    # identifikasi kelas akibat nilai kanal RGB yang kebetulan sama.
    kernel = np.ones((7, 7), np.uint8)
    class_masks = {
        label: np.zeros((h, w), dtype=bool)
        for label in COLORS
    }
    objek_terdeteksi = {}

    for class_id in np.unique(pred_seg):
        label = remap_class(int(class_id))
        if label is None:
            continue

        mask = (pred_seg == class_id).astype(np.uint8)
        if np.sum(mask) < 500:
            continue

        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        # Gabungkan class ID yang dipetakan ke kelompok analisis yang sama.
        class_masks[label] |= mask.astype(bool)
        objek_terdeteksi[label] = COLORS[label]

    # Morfologi dapat membuat mask kelas yang semula berbeda saling tumpang tindih.
    # Hilangkan overlap secara deterministik berdasarkan urutan prioritas berikut.
    # Mask analitis akhir menjadi saling eksklusif; mask integer tidak bergantung warna.
    priority = [
        "building", "vegetation", "sky", "ground",
        "vehicle_4w", "vehicle_2w", "human", "traffic_sign"
    ]
    occupied = np.zeros((h, w), dtype=bool)
    for label in priority:
        if label not in class_masks:
            continue
        class_masks[label] = class_masks[label] & (~occupied)
        occupied |= class_masks[label]

    # Citra RGB untuk visualisasi saja; jangan gunakan warna ini untuk menghitung kelas.
    mask_final = np.zeros((h, w, 3), dtype=np.uint8)
    for label in priority:
        if label in objek_terdeteksi:
            mask_final[class_masks[label]] = np.asarray(COLORS[label], dtype=np.uint8)

    # ===============================
    # 8. OVERLAY
    # ===============================
    overlay = Image.blend(
        image.convert("RGBA"),
        Image.fromarray(mask_final).convert("RGBA"),
        alpha=0.6
    )

    # ===============================
    # 9. VISUALISASI (SAMA PERSIS)
    # ===============================
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))

    axes[0].set_title("Input")
    axes[0].imshow(image)
    axes[0].axis('off')

    axes[1].set_title("Segmentasi Clean")
    axes[1].imshow(mask_final)
    axes[1].axis('off')

    axes[2].set_title("Overlay")
    axes[2].imshow(overlay)
    axes[2].axis('off')

    # legend
    patches = []
    for label, warna in objek_terdeteksi.items():
        patches.append(mpatches.Patch(color=[w/255 for w in warna], label=label))

    plt.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    # ===============================
    # 10. ANALISIS
    # ===============================
    print("\nHASIL ANALISIS:")
    print("-"*40)

    for label in objek_terdeteksi:
        mask = class_masks[label]
        persen = np.sum(mask) / mask.size * 100
        print(f"{label.ljust(15)} : {persen:.2f}%")

    # =====================================
    # 10A. COMPOSITE VISUAL INDICATORS
    # =====================================

    visual_profile = {

        "Building Visibility":0,
        "Vegetation Coverage":0,
        "Sky Openness":0,
        "Ground Accessibility":0,
        "Human Activity":0,
        "Vehicle Intensity":0,
        "Traffic Infrastructure":0,
        "Heritage Dominance":0

    }

    for label in objek_terdeteksi:

        mask = class_masks[label]

        persen = np.sum(mask) / mask.size * 100

        if label == "building":
            visual_profile["Building Visibility"] = persen

            # ------------------------------
            # Heritage Dominance
            # TANPA ROI
            # diasumsikan bangunan utama
            # pada foto adalah heritage
            # ------------------------------
            visual_profile["Heritage Dominance"] = persen

        elif label == "vegetation":
            visual_profile["Vegetation Coverage"] = persen

        elif label == "sky":
            visual_profile["Sky Openness"] = persen

        elif label == "ground":
            visual_profile["Ground Accessibility"] = persen

        elif label == "human":
            visual_profile["Human Activity"] = persen

        elif label == "vehicle_4w":

            visual_profile["Vehicle Intensity"] = persen

        elif label == "vehicle_2w":

            visual_profile["Vehicle Intensity"] += persen

        elif label == "traffic_sign":

            visual_profile["Traffic Infrastructure"] = persen

    print("\n")
    print("="*45)
    print("COMPOSITE VISUAL INDICATORS")
    print("="*45)

    for k,v in visual_profile.items():

        print(f"{k.ljust(25)} : {v:.2f}%")

    # ===============================
    # 10B. MORPHOLOGICAL FEATURES
    # ===============================

    import cv2

    morph_features = {}

    for label in objek_terdeteksi:

        mask = class_masks[label].astype(np.uint8)

        contours,_ = cv2.findContours(
            mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        if len(contours)==0:
            continue

        c = max(contours,key=cv2.contourArea)

        area = cv2.contourArea(c)

        perimeter = cv2.arcLength(c,True)

        hull = cv2.convexHull(c)

        hull_area = cv2.contourArea(hull)

        solidity = area/hull_area if hull_area>0 else 0

        morph_features[label] = {

            "Area":area,

            "Perimeter":perimeter,

            "Solidity":solidity

        }

    # ===============================
    # COLOR FEATURES
    # ===============================

    color_features = {}

    for label in objek_terdeteksi:

        mask = class_masks[label]

        pixels = img[mask]

        if len(pixels)==0:
            continue

        mean_rgb = np.mean(pixels,axis=0)

        color_features[label]={

            "MeanR":mean_rgb[0],

            "MeanG":mean_rgb[1],

            "MeanB":mean_rgb[2]

        }

    # scikit-image seharusnya sudah terpasang pada sel instalasi awal.

    # =====================================
    # TEXTURE FEATURES (GLCM)
    # =====================================

    from skimage.feature import graycomatrix, graycoprops

    texture_features = {}

    # grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    for label in objek_terdeteksi:

        mask = class_masks[label]

        if np.sum(mask) < 100:
            continue

        # hanya area objek
        gray_object = gray.copy()
        gray_object[~mask] = 0

        # GLCM
        glcm = graycomatrix(
            gray_object,
            distances=[1],
            angles=[0],
            levels=256,
            symmetric=True,
            normed=True
        )

        contrast = graycoprops(glcm, 'contrast')[0, 0]
        correlation = graycoprops(glcm, 'correlation')[0, 0]
        energy = graycoprops(glcm, 'energy')[0, 0]
        homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
        asm = graycoprops(glcm, 'ASM')[0, 0]

        texture_features[label] = {

            "Contrast": contrast,

            "Correlation": correlation,

            "Energy": energy,

            "Homogeneity": homogeneity,

            "ASM": asm

        }

    print("\nTEXTURE FEATURES")
    print("=" * 45)

    for label in texture_features:

        print(f"\n{label}")

        for k, v in texture_features[label].items():

            print(f"   {k:15s}: {v:.4f}")

    # ---------------------------------------
    # Semantic Ratios
    # ---------------------------------------

    pixel_ratio = {}

    for label in objek_terdeteksi:

        mask = class_masks[label]

        pixel_ratio[label] = np.sum(mask) / mask.size

    # =====================================
    # VISUAL PROFILE (0-1)
    # =====================================

    visual_profile = {

        "Building Visibility":
            pixel_ratio.get("building",0),

        "Vegetation Coverage":
            pixel_ratio.get("vegetation",0),

        "Sky Openness":
            pixel_ratio.get("sky",0),

        "Ground Accessibility":
            pixel_ratio.get("ground",0),

        "Human Activity":
            pixel_ratio.get("human",0),

        "Vehicle Intensity":
            pixel_ratio.get("vehicle_4w",0) +
            pixel_ratio.get("vehicle_2w",0),

        "Traffic Infrastructure":
            pixel_ratio.get("traffic_sign",0),

        "Heritage Dominance":
            pixel_ratio.get("building",0)

    }
    import pandas as pd

    visual_profile_df = pd.DataFrame(

        list(visual_profile.items()),

        columns=[
            "Visual Indicator",
            "Value (0–1)"
        ]

    )

    display(visual_profile_df)

    labels = list(visual_profile.keys())

    values = list(visual_profile.values())

    # ==================================================
    # 11. COMPOSITE VISUAL INDICES
    # Theoretical Weighting Framework
    # ==================================================

    visual_indices = {}

    # ---------------------------------------
    # Helper Function
    # ---------------------------------------

    def normalize(value, max_value):

        if max_value == 0:
            return 0

        return min(value / max_value, 1.0)


    # ==================================================
    # Building Visibility Index
    # ==================================================

    building_ratio = pixel_ratio.get("building",0)

    building_area = morph_features.get(
        "building",{}
    ).get("Area",0)

    building_solidity = morph_features.get(
        "building",{}
    ).get("Solidity",0)

    building_area = normalize(
        building_area,
        h*w
    )

    BVI = (

        0.50 * building_ratio +

        0.30 * building_area +

        0.20 * building_solidity

    )

    visual_indices["Building Visibility Index"] = BVI

    # ==================================================
    # Vegetation Coverage Index
    # ==================================================

    vegetation_ratio = pixel_ratio.get(
        "vegetation",
        0
    )

    vegetation_texture = texture_features.get(
        "vegetation",
        {}
    ).get(
        "Homogeneity",
        0
    )

    VCI = (

        0.70 * vegetation_ratio +

        0.30 * vegetation_texture

    )

    visual_indices["Vegetation Coverage Index"] = VCI

    # ==================================================
    # Sky Openness Index
    # ==================================================

    sky_ratio = pixel_ratio.get("sky",0)

    sky_texture = texture_features.get(
        "sky",{}
    ).get(
        "Homogeneity",
        0
    )

    SOI = (

        0.80 * sky_ratio +

        0.20 * sky_texture

    )

    visual_indices["Sky Openness Index"] = SOI

    # ==================================================
    # Ground Accessibility Index
    # ==================================================

    ground_ratio = pixel_ratio.get(
        "ground",
        0
    )

    ground_solidity = morph_features.get(
        "ground",
        {}
    ).get(
        "Solidity",
        0
    )

    GAI = (

        0.70 * ground_ratio +

        0.30 * ground_solidity

    )

    visual_indices["Ground Accessibility Index"] = GAI

    # ==================================================
    # Human Activity Index
    # ==================================================

    human_ratio = pixel_ratio.get(
        "human",
        0
    )

    HAI = human_ratio

    visual_indices["Human Activity Index"] = HAI

    # ==================================================
    # Vehicle Intensity Index
    # ==================================================

    vehicle_ratio = (

        pixel_ratio.get("vehicle_4w",0)

        +

        pixel_ratio.get("vehicle_2w",0)

    )

    VII = vehicle_ratio

    visual_indices["Vehicle Intensity Index"] = VII

    # ==================================================
    # Traffic Infrastructure Index
    # ==================================================

    traffic_ratio = pixel_ratio.get(
        "traffic_sign",
        0
    )

    TII = traffic_ratio

    visual_indices["Traffic Infrastructure Index"] = TII

    # ==================================================
    # Heritage Dominance Index
    # ==================================================

    building_rgb = np.mean([

        color_features.get(
            "building",
            {}
        ).get("MeanR",0),

        color_features.get(
            "building",
            {}
        ).get("MeanG",0),

        color_features.get(
            "building",
            {}
        ).get("MeanB",0)

    ]) / 255

    texture_score = texture_features.get(
        "building",
        {}
    ).get(
        "Homogeneity",
        0
    )

    HDI = (

        0.40 * BVI +

        0.25 * building_solidity +

        0.20 * building_rgb +

        0.15 * texture_score

    )

    visual_indices["Heritage Dominance Index"] = HDI

    # ==================================================
    # Print Composite Visual Indices
    # ==================================================

    print("\\n")
    print("="*80)
    print("COMPOSITE VISUAL INDICES")
    print("="*80)

    for key, value in visual_indices.items():
        print(f"{key:35s}: {value:.4f}")

    # ==================================================
    # 11A. THEORETICAL WEIGHTING & OVERALL COMPOSITE SCORE
    # ==================================================
    # Bobot teoretis sesuai tabel; jumlah bobot = 1.00 (100%).
    theoretical_weights = {
        "Building Visibility Index": 0.20,
        "Vegetation Coverage Index": 0.15,
        "Sky Openness Index": 0.10,
        "Ground Accessibility Index": 0.10,
        "Human Activity Index": 0.10,
        "Vehicle Intensity Index": 0.05,
        "Traffic Infrastructure Index": 0.05,
        "Heritage Dominance Index": 0.25,
    }

    # Nilai tertimbang setiap indeks = nilai indeks x bobot teoretis.
    weighted_visual_indices = {
        name: visual_indices[name] * weight
        for name, weight in theoretical_weights.items()
    }

    # Skor akhir merupakan jumlah seluruh nilai tertimbang.
    overall_composite_visual_index = sum(weighted_visual_indices.values())
    weight_total = sum(theoretical_weights.values())

    print("\\n" + "="*90)
    print("THEORETICAL WEIGHTING - COMPOSITE VISUAL INDEX")
    print("="*90)
    print(f"{'Composite Visual Index':38s} {'Index Value':>12s} {'Weight':>10s} {'Weighted Value':>16s}")
    print("-"*90)
    for name, weight in theoretical_weights.items():
        print(f"{name:38s} {visual_indices[name]:12.4f} {weight:10.2f} {weighted_visual_indices[name]:16.4f}")
    print("-"*90)
    print(f"{'TOTAL':38s} {'':12s} {weight_total:10.2f} {overall_composite_visual_index:16.4f}")
    print(f"Overall Composite Visual Index (0-1): {overall_composite_visual_index:.4f}")
    print(f"Overall Composite Visual Index (0-100): {overall_composite_visual_index*100:.2f}")

    # ======================================
    # RADAR CHART COMPOSITE VISUAL INDICES
    # ======================================

    labels = list(visual_indices.keys())
    values = list(visual_indices.values())

    angles = np.linspace(
        0,
        2*np.pi,
        len(labels),
        endpoint=False
    )

    angles = np.concatenate((angles,[angles[0]]))
    values = values + [values[0]]

    fig = plt.figure(figsize=(8,8))

    ax = plt.subplot(111, polar=True)

    ax.plot(
        angles,
        values,
        linewidth=2,
        color="darkred"
    )

    ax.fill(
        angles,
        values,
        alpha=0.30,
        color="red"
    )

    ax.set_xticks(angles[:-1])

    ax.set_xticklabels(labels)

    ax.set_ylim(0,1)

    ax.set_title(
        "Composite Visual Indices",
        fontsize=16,
        fontweight='bold',
        pad=35,
        y=1.12
    )
    plt.tight_layout()

    plt.subplots_adjust(top=0.88)
    plt.show()

    # ===============================
    # 12. DIAGRAM BATANG (KOMPOSISI ELEMEN VISUAL)
    # ===============================
    # Diagram ini khusus untuk komposisi kelas (persen), bukan indeks komposit.
    COLOR_MAP = {
        "ground": [150/255, 100/255, 100/255],
        "building": [70/255, 70/255, 70/255],
        "traffic_sign": [220/255, 220/255, 0],
        "vegetation": [107/255, 142/255, 35/255],
        "sky": [70/255, 130/255, 180/255],
        "human": [220/255, 20/255, 60/255],
        "vehicle_4w": [0, 0, 142/255],
        "vehicle_2w": [0, 0, 230/255],
    }

    composition = []
    for label in objek_terdeteksi:
        ratio = pixel_ratio.get(label, 0.0)
        composition.append((ratio * 100, label, COLOR_MAP[label]))

    composition.sort(key=lambda item: item[0], reverse=True)
    bar_values = [item[0] for item in composition]
    bar_labels = [item[1] for item in composition]
    bar_colors = [item[2] for item in composition]

    plt.figure(figsize=(8, 5))
    plt.barh(bar_labels, bar_values, color=bar_colors)
    plt.xlabel("Persentase (%)")
    plt.title("Komposisi Elemen Visual")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

    # =====================================
    # 13. RADAR CHART
    # =====================================

    import numpy as np
    import matplotlib.pyplot as plt

    labels = list(visual_profile.keys())

    values = list(visual_profile.values())

    angles = np.linspace(
        0,
        2*np.pi,
        len(labels),
        endpoint=False
    )

    angles = np.concatenate((angles,[angles[0]]))
    values = values + [values[0]]

    fig = plt.figure(figsize=(8,8))

    ax = plt.subplot(111,polar=True)

    ax.plot(
        angles,
        values,
        linewidth=2,
        color="darkred"
    )

    ax.fill(
        angles,
        values,
        alpha=0.30,
        color="red"
    )

    ax.set_xticks(angles[:-1])

    ax.set_xticklabels(labels)

    ax.set_ylim(0,max(values)*1.15)

    ax.set_title(
        "Urban Visual Quality Profile",
        fontsize=14,
        pad=20
    )

    plt.tight_layout()

    plt.show()

    # ============================================================
    # 14. EXPORT FULL ANALYSIS TO PDF
    # ============================================================

    from matplotlib.backends.backend_pdf import PdfPages
    from matplotlib.figure import Figure
    import matplotlib.pyplot as plt
    import numpy as np
    import os

    safe_stem = os.path.splitext(os.path.basename(image_path))[0]
    pdf_filename = f"FULL_ANALYSIS_URBAN_VISUAL_QUALITY_{safe_stem}.pdf"

    # ============================================================
    # FUNGSI UNTUK MEMBUAT HALAMAN JUDUL
    # ============================================================

    def add_title_page(pdf):

        fig = plt.figure(figsize=(8.27, 11.69))  # A4 portrait
        ax = fig.add_subplot(111)

        ax.axis("off")

        ax.text(
            0.5,
            0.72,
            "URBAN VISUAL QUALITY ANALYSIS",
            ha="center",
            va="center",
            fontsize=22,
            fontweight="bold"
        )

        ax.text(
            0.5,
            0.62,
            "AI-Based Semantic Segmentation",
            ha="center",
            va="center",
            fontsize=15
        )

        ax.text(
            0.5,
            0.55,
            "Composite Visual Indicators & Visual Indices",
            ha="center",
            va="center",
            fontsize=13
        )

        ax.text(
            0.5,
            0.45,
            f"Image: {image_path}",
            ha="center",
            va="center",
            fontsize=12
        )

        ax.text(
            0.5,
            0.38,
            "SegFormer – Cityscapes",
            ha="center",
            va="center",
            fontsize=11
        )

        pdf.savefig(
            fig,
            bbox_inches="tight"
        )

        plt.close(fig)


    # ============================================================
    # FUNGSI UNTUK MENYIMPAN GAMBAR
    # ============================================================

    def add_image_page(
        pdf,
        img,
        title,
        figsize=(11.69, 8.27)
    ):

        fig = plt.figure(figsize=figsize)

        ax = fig.add_subplot(111)

        ax.imshow(img)

        ax.axis("off")

        ax.set_title(
            title,
            fontsize=16,
            fontweight="bold",
            pad=15
        )

        pdf.savefig(
            fig,
            bbox_inches="tight"
        )

        plt.close(fig)


    # ============================================================
    # FUNGSI UNTUK MENYIMPAN TABEL / TEKS
    # ============================================================

    def add_text_page(
        pdf,
        title,
        text
    ):

        fig = plt.figure(
            figsize=(8.27, 11.69)
        )

        ax = fig.add_subplot(111)

        ax.axis("off")

        ax.set_title(
            title,
            fontsize=17,
            fontweight="bold",
            pad=20
        )

        ax.text(
            0.05,
            0.95,
            text,
            transform=ax.transAxes,
            fontsize=11,
            verticalalignment="top",
            family="monospace"
        )

        pdf.savefig(
            fig,
            bbox_inches="tight"
        )

        plt.close(fig)


    # ============================================================
    # MULAI MEMBUAT PDF
    # ============================================================

    with PdfPages(pdf_filename) as pdf:

        # --------------------------------------------------------
        # 1. TITLE PAGE
        # --------------------------------------------------------

        add_title_page(pdf)


        # --------------------------------------------------------
        # 2. INPUT IMAGE
        # --------------------------------------------------------

        add_image_page(
            pdf,
            image,
            "1. INPUT IMAGE"
        )


        # --------------------------------------------------------
        # 3. SEGMENTASI CLEAN
        # --------------------------------------------------------

        add_image_page(
            pdf,
            mask_final,
            "2. AI SEMANTIC SEGMENTATION – CLEAN"
        )


        # --------------------------------------------------------
        # 4. OVERLAY
        # --------------------------------------------------------

        add_image_page(
            pdf,
            overlay,
            "3. SEMANTIC SEGMENTATION OVERLAY"
        )


        # --------------------------------------------------------
        # 5. HASIL ANALISIS PERSENTASE
        # --------------------------------------------------------

        percentage_text = ""

        percentage_text += (
            "HASIL ANALISIS ELEMEN VISUAL\n"
            "========================================\n\n"
        )

        for label in objek_terdeteksi:

            mask = class_masks[label]

            persen = (
                np.sum(mask)
                / mask.size
                * 100
            )

            percentage_text += (
                f"{label:<20} : "
                f"{persen:>8.2f}%\n"
            )

        add_text_page(
            pdf,
            "4. HASIL ANALISIS ELEMEN VISUAL",
            percentage_text
        )


        # --------------------------------------------------------
        # 6. COMPOSITE VISUAL INDICATORS
        # --------------------------------------------------------

        profile_text = ""

        profile_text += (
            "COMPOSITE VISUAL INDICATORS\n"
            "========================================\n\n"
        )

        for key, value in visual_profile.items():

            profile_text += (
                f"{key:<30} : "
                f"{value:.2f}%\n"
            )

        add_text_page(
            pdf,
            "5. COMPOSITE VISUAL INDICATORS",
            profile_text
        )


        # --------------------------------------------------------
        # 7. VISUAL PROFILE TABLE
        # --------------------------------------------------------

        fig = plt.figure(
            figsize=(8.27, 11.69)
        )

        ax = fig.add_subplot(111)

        ax.axis("off")

        table_data = visual_profile_df.copy()

        table = ax.table(
            cellText=table_data.values,
            colLabels=table_data.columns,
            loc="center",
            cellLoc="center"
        )

        table.auto_set_font_size(False)

        table.set_fontsize(10)

        table.scale(
            1,
            2
        )

        ax.set_title(
            "6. VISUAL PROFILE TABLE",
            fontsize=17,
            fontweight="bold",
            pad=30
        )

        pdf.savefig(
            fig,
            bbox_inches="tight"
        )

        plt.close(fig)


        # --------------------------------------------------------
        # 8. MORPHOLOGICAL FEATURES
        # --------------------------------------------------------

        morph_text = ""

        morph_text += (
            "MORPHOLOGICAL FEATURES\n"
            "========================================\n\n"
        )

        for label, data in morph_features.items():

            morph_text += f"{label.upper()}\n"

            for key, value in data.items():

                morph_text += (
                    f"   {key:<15} : "
                    f"{value:.4f}\n"
                )

            morph_text += "\n"

        add_text_page(
            pdf,
            "7. MORPHOLOGICAL FEATURES",
            morph_text
        )


        # --------------------------------------------------------
        # 9. COLOR FEATURES
        # --------------------------------------------------------

        color_text = ""

        color_text += (
            "COLOR FEATURES\n"
            "========================================\n\n"
        )

        for label, data in color_features.items():

            color_text += f"{label.upper()}\n"

            for key, value in data.items():

                color_text += (
                    f"   {key:<15} : "
                    f"{value:.4f}\n"
                )

            color_text += "\n"

        add_text_page(
            pdf,
            "8. COLOR FEATURES",
            color_text
        )


        # --------------------------------------------------------
        # 10. TEXTURE FEATURES
        # --------------------------------------------------------

        texture_text = ""

        texture_text += (
            "TEXTURE FEATURES – GLCM\n"
            "========================================\n\n"
        )

        for label, data in texture_features.items():

            texture_text += f"{label.upper()}\n"

            for key, value in data.items():

                texture_text += (
                    f"   {key:<15} : "
                    f"{value:.4f}\n"
                )

            texture_text += "\n"

        add_text_page(
            pdf,
            "9. TEXTURE FEATURES – GLCM",
            texture_text
        )


        # --------------------------------------------------------
        # --------------------------------------------------------
        # 11. COMPOSITE VISUAL INDICES + THEORETICAL WEIGHTING
        # --------------------------------------------------------

        index_text = ""
        index_text += (
            "COMPOSITE VISUAL INDICES\\n"
            "========================================\\n\\n"
        )

        for key, value in visual_indices.items():
            index_text += f"{key:<35} : {value:.4f}\\n"

        index_text += (
            "\\nTHEORETICAL WEIGHTING\\n"
            "------------------------------------------------------------\\n"
            f"{'Index':<35} {'Value':>8} {'Weight':>8} {'Weighted':>10}\\n"
        )
        for key, weight in theoretical_weights.items():
            index_text += (
                f"{key:<35} "
                f"{visual_indices[key]:>8.4f} "
                f"{weight:>8.2f} "
                f"{weighted_visual_indices[key]:>10.4f}\\n"
            )

        index_text += (
            "------------------------------------------------------------\\n"
            f"{'TOTAL WEIGHT':<35} {'':>8} {weight_total:>8.2f} "
            f"{overall_composite_visual_index:>10.4f}\\n"
            f"OVERALL COMPOSITE VISUAL INDEX (0-1): "
            f"{overall_composite_visual_index:.4f}\\n"
            f"OVERALL COMPOSITE VISUAL INDEX (0-100): "
            f"{overall_composite_visual_index*100:.2f}\\n"
        )

        add_text_page(
            pdf,
            "10. COMPOSITE VISUAL INDICES & THEORETICAL WEIGHTING",
            index_text
        )


        # 12. RADAR COMPOSITE VISUAL INDICES
        # --------------------------------------------------------

        labels = list(
            visual_indices.keys()
        )

        values = list(
            visual_indices.values()
        )

        angles = np.linspace(
            0,
            2 * np.pi,
            len(labels),
            endpoint=False
        )

        angles = np.concatenate(
            [angles, [angles[0]]]
        )

        values_radar = values + [
            values[0]
        ]

        fig = plt.figure(
            figsize=(8.27, 8.27)
        )

        ax = fig.add_subplot(
            111,
            polar=True
        )

        ax.plot(
            angles,
            values_radar,
            linewidth=2
        )

        ax.fill(
            angles,
            values_radar,
            alpha=0.30
        )

        ax.set_xticks(
            angles[:-1]
        )

        ax.set_xticklabels(
            labels,
            fontsize=9
        )

        ax.set_ylim(
            0,
            1
        )

        ax.set_title(
            "11. Composite Visual Indices",
            fontsize=16,
            fontweight="bold",
            pad=25
        )

        pdf.savefig(
            fig,
            bbox_inches="tight"
        )

        plt.close(fig)


        # --------------------------------------------------------
        # 13. DIAGRAM BATANG VISUAL INDICES
        # --------------------------------------------------------

        sorted_indices = sorted(
            visual_indices.items(),
            key=lambda x: x[1],
            reverse=True
        )

        bar_labels = [
            x[0]
            for x in sorted_indices
        ]

        bar_values = [
            x[1]
            for x in sorted_indices
        ]

        fig = plt.figure(
            figsize=(11.69, 8.27)
        )

        ax = fig.add_subplot(111)

        ax.barh(
            bar_labels,
            bar_values
        )

        ax.set_xlabel(
            "Index Value (0–1)"
        )

        ax.set_title(
            "12. Composite Visual Indices – Bar Chart",
            fontsize=16,
            fontweight="bold"
        )

        ax.set_xlim(
            0,
            1
        )

        ax.invert_yaxis()

        plt.tight_layout()

        pdf.savefig(
            fig,
            bbox_inches="tight"
        )

        plt.close(fig)


        # --------------------------------------------------------
        # 14. URBAN VISUAL QUALITY PROFILE
        # --------------------------------------------------------

        labels = list(
            visual_profile.keys()
        )

        values = list(
            visual_profile.values()
        )

        angles = np.linspace(
            0,
            2 * np.pi,
            len(labels),
            endpoint=False
        )

        angles = np.concatenate(
            [angles, [angles[0]]]
        )

        values_radar = values + [
            values[0]
        ]

        fig = plt.figure(
            figsize=(8.27, 8.27)
        )

        ax = fig.add_subplot(
            111,
            polar=True
        )

        ax.plot(
            angles,
            values_radar,
            linewidth=2
        )

        ax.fill(
            angles,
            values_radar,
            alpha=0.30
        )

        ax.set_xticks(
            angles[:-1]
        )

        ax.set_xticklabels(
            labels,
            fontsize=9
        )

        max_value = max(
            values
        )

        ax.set_ylim(
            0,
            max_value * 1.15
            if max_value > 0
            else 1
        )

        ax.set_title(
            "13. Urban Visual Quality Profile",
            fontsize=16,
            fontweight="bold",
            pad=25
        )

        pdf.savefig(
            fig,
            bbox_inches="tight"
        )

        plt.close(fig)


    # ============================================================
    # SELESAI
    # ============================================================

    print("\n")
    print("=" * 70)
    print("PDF FULL ANALYSIS BERHASIL DIBUAT")
    print("=" * 70)
    print(
        f"File : {pdf_filename}"
    )
    print(
        f"Lokasi : {os.path.abspath(pdf_filename)}"
    )
    print("=" * 70)
    generated_pdfs.append(pdf_filename)

# ============================================================
# BUNDLING HASIL PDF SELURUH FOTO
# ============================================================
zip_filename = "HASIL_ANALISIS_PVI_MULTI_FOTO.zip"
with zipfile.ZipFile(zip_filename, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    for pdf_path in generated_pdfs:
        if os.path.exists(pdf_path):
            zipf.write(pdf_path, arcname=os.path.basename(pdf_path))

print("\\nSelesai. PDF dibuat untuk", len(generated_pdfs), "foto.")
print("Paket hasil:", zip_filename)
files.download(zip_filename)
